# Day3: Exploratory Analysis

このノートブックでは、正規化したRNA-seqデータを PCA とヒートマップで眺める方法を学びます。

このトピックが役立つ場面:
差次的発現解析に進む前に、炎症刺激群と対照群が全体として分かれているか、外れ値がないかを確認したい場面です。

このトピックで解く課題:
このデータは刺激応答をちゃんと持っていそうか、それとも外れ値やノイズが強そうかを可視化で見極めます。

## キーワード辞典（この章）

| キーワード | まずこう理解する | この章での使いどころ | よくある誤解 |
|---|---|---|---|
| PCA | 高次元を少数軸に圧縮 | サンプル全体構造の確認 | PC1/PC2に生物学名を即断で付ける |
| principal component | 分散を説明する軸 | 群分離や外れ値確認 | 因果関係の軸と誤解する |
| heatmap | 行列の強弱可視化 | サンプル類似性の確認 | 色の印象だけで結論を出す |
| clustering | 似たものをまとめる | replicate整合性確認 | クラスタ数を固定で正しいと考える |
| outlier | 他サンプルから外れる点 | ラベル誤り・品質問題の発見 | 必ず除外すべきと短絡する |



## この章での判断軸

1. 群分離が見えても、まずmetadata整合性を確認する。
2. 外れ値は原因を説明できるまで保留する。
3. 可視化は結論ではなく、次の検証を決める手がかりとして使う。


## 探索的解析の役割

探索的解析は、統計解析の前にデータ全体を眺める工程です。これは、地図を見ずに目的地へ向かわないのと同じです。

RNA-seqでは、各サンプルは数千から数万の遺伝子発現量を持つ高次元データです。そのままでは人間の目で比較できません。PCAやヒートマップは、この高次元データを人間が読める形に圧縮するための道具です。

ここで見たいのは、個別の遺伝子ではなく、サンプル全体の関係です。

## なぜ探索的解析をするのか

差次的発現解析に進む前に、まずデータ全体の構造を見ることが重要です。条件ごとにサンプルがまとまっているか、極端に離れたサンプルがないかを可視化で確認します。

In [ ]:
# この章では、正規化済みデータを可視化して全体像を見ます。
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# まずcount matrixを用意し、Day2と同じ流れでlog-CPMを作ります。
counts = pd.DataFrame({
    "gene": ["TP53", "MYC", "GAPDH", "ACTB", "IL6"],
    "control_1": [120, 300, 5000, 4200, 30],
    "control_2": [100, 280, 5100, 4000, 25],
    "treated_1": [300, 600, 5300, 4100, 200],
    "treated_2": [280, 650, 5200, 4300, 220],
}).set_index("gene")

# PCAやヒートマップには、生のcountではなく正規化後の値を使います。
library_size = counts.sum(axis=0)
cpm = counts.div(library_size, axis=1) * 1_000_000
log_cpm = np.log2(cpm + 1)
log_cpm

## PCA をどう読むか

PCA は、たくさんの遺伝子からなる発現パターンを、少数の軸に要約します。PC1やPC2は、元の遺伝子名そのものではありません。多くの遺伝子の変動をまとめた新しい軸です。

PCAプロットでは、点がサンプルを表します。点同士が近ければ、発現パターンが似ています。点同士が遠ければ、発現パターンが違います。

この教材では、control同士が近く、treated同士が近く、さらにcontrolとtreatedが分かれて見えるかを確認します。そう見えれば、刺激による全体的な発現変化がある可能性があります。

## PCA

PCA は、多次元の発現データを少数の軸に圧縮して、サンプル間の近さを見やすくする方法です。近いサンプルほど発現パターンが似ています。

In [ ]:
# サンプルを点として描くため、遺伝子 x サンプルの表を転置します。
# PCAは各サンプルを、発現パターンを要約した2つの軸に配置します。
pca = PCA(n_components=2)
coords = pca.fit_transform(log_cpm.T)
pca_df = pd.DataFrame(coords, columns=["PC1", "PC2"], index=log_cpm.columns)
pca_df["condition"] = ["control", "control", "treated", "treated"]
pca_df

In [ ]:
# PCAプロットでは、近い点ほど発現パターンが似ています。
# control同士、treated同士が近いかを見ます。
plt.figure(figsize=(5, 4))
colors = {"control": "#4C78A8", "treated": "#F58518"}
for sample, row in pca_df.iterrows():
    plt.scatter(row["PC1"], row["PC2"], color=colors[row["condition"]], s=80)
    plt.text(row["PC1"] + 0.02, row["PC2"] + 0.02, sample)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of log-CPM")
plt.tight_layout()
plt.show()

## ヒートマップをどう読むか

ヒートマップは、遺伝子ごとの発現量を色で表す図です。PCAがサンプル同士の距離を見る図だとすると、ヒートマップは「どの遺伝子がどのサンプルで高いか」を見る図です。

色が濃い場所は値が高く、薄い場所は値が低いことを示します。`IL6` の行が treated 側で高く見えれば、炎症刺激に反応している候補として自然に読めます。

ただし、ヒートマップは見た目の確認です。最終的な判断には、次章の差次的発現解析が必要です。

## ヒートマップ的に見る

ここでは `matplotlib` だけで簡単な発現ヒートマップを描きます。色の違いで、どの遺伝子がどのサンプルで高いかを直感的に見られます。

In [ ]:
# ヒートマップでは、遺伝子ごとの発現量を色で表します。
# 行方向に遺伝子、列方向にサンプルが並びます。
plt.figure(figsize=(6, 4))
plt.imshow(log_cpm.values, aspect="auto", cmap="viridis")
plt.colorbar(label="log2(CPM + 1)")
plt.xticks(range(len(log_cpm.columns)), log_cpm.columns, rotation=30, ha="right")
plt.yticks(range(len(log_cpm.index)), log_cpm.index)
plt.title("Gene expression heatmap")
plt.tight_layout()
plt.show()

## どう読むか

- PCAで control 同士、treated 同士が近ければ、条件差が反映されている可能性があります。
- ヒートマップで `IL6` が treated 側で高ければ、条件差を視覚的にも確認できます。
- もし1サンプルだけ極端に離れていれば、外れ値や実験上の問題も疑います。

## ここまでの要点

- 探索的解析は、差次的発現解析の前にデータを眺める工程
- PCA はサンプル間の近さを見る
- ヒートマップは遺伝子ごとの発現パターンを見る
- 次は統計的に「どの遺伝子が変化したか」を考える